In [2]:
!pip install tqdm

In [52]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import math

## 表9.10の再現

In [4]:
# 表9.4のデータセット
df = pd.DataFrame(
    {"HNK": [22, 16, 19, 11], "TNK": [2, 54, 33, 17], "EXT": [10, 115, 73, 28]},
    index=["HMF", "SSM", "NOD", "IND"],
)
df

,HNK,TNK,EXT
HMF,22,2,10
SSM,16,54,115
NOD,19,33,73
IND,11,17,28


In [5]:
# セルごとの期待度数を計算（加法モデルの場合）
theta_k = (df.sum(axis=0) / df.sum().sum()).values
E_y_additional = (df.sum(axis=1).values.reshape(4, 1) * theta_k).flatten()
E_y_additional

array([  5.78 ,   9.01 ,  19.21 ,  31.45 ,  49.025, 104.525,  21.25 ,
        33.125,  70.625,   9.52 ,  14.84 ,  31.64 ])

In [6]:
# セルごとの期待度数を計算（飽和モデルの場合）
# 飽和モデルの場合E_yの推定値は実績値と等しい
# theta_jk  = (df / df.sum().sum()).values
# E_y_saturated = (df.sum().sum() * theta_jk).flatten()
E_y_saturated = df.values.flatten()
E_y_saturated

array([ 22,   2,  10,  16,  54, 115,  19,  33,  73,  11,  17,  28])

In [7]:
# セルごとの期待度数を計算（最小モデルの場合）
E_y_min = df.sum().sum() / (df.size)
E_y_min

33.333333333333336

In [8]:
# 期待度数の対数変換が目的変数
log_E_y_additional = np.log(E_y_additional)

# 期待度数の対数変換が目的変数
log_E_y_saturated = np.log(E_y_saturated)

# 期待度数の対数変換が目的変数
log_E_y_min = np.array([np.log(np.array(E_y_min))])

In [12]:
# 加法モデルの場合のデザイン行列
# 基準カテゴリーがHMFとHNKなので1行目と1列目の項は0とみなす2元配置分散分析モデル
X_additive = np.array(
    [
        [1, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 1, 0],
        [1, 0, 0, 0, 0, 1],
        [1, 1, 0, 0, 0, 0],
        [1, 1, 0, 0, 1, 0],
        [1, 1, 0, 0, 0, 1],
        [1, 0, 1, 0, 0, 0],
        [1, 0, 1, 0, 1, 0],
        [1, 0, 1, 0, 0, 1],
        [1, 0, 0, 1, 0, 0],
        [1, 0, 0, 1, 1, 0],
        [1, 0, 0, 1, 0, 1],
    ]
)

# 6.4.2節と同様の処理（最小二乗法）により係数を推定
b_additive = (
    (np.linalg.inv(X_additive.T @ X_additive)) @ X_additive.T @ log_E_y_additional
)

In [13]:
# 飽和モデル
X_saturated = np.array(
    [
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
        [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0],
        [1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0],
        [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0],
        [1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
    ]
)
b_saturated = (
    np.linalg.inv(X_saturated.T @ X_saturated) @ X_saturated.T @ log_E_y_saturated
)

In [14]:
table_910 = pd.DataFrame(
    {
        "飽和モデル(9.11)": np.round(b_saturated, 3),
        "加法モデル(9.10)": np.round(b_additive, 3).tolist()
        + ["-"] * (len(b_saturated) - len(b_additive)),
        "最小モデル": np.round(log_E_y_min, 3).tolist()
        + ["-"] * (len(b_saturated) - len(log_E_y_min)),
    },
    index=[
        "定数",
        "SSM",
        "NOD",
        "IND",
        "TNK",
        "EXT",
        "SSM*TNK",
        "SSM*EXT",
        "NOD*TNK",
        "NOD*EXT",
        "IND*TNK",
        "IND*EXT",
    ],
)
table_910

,飽和モデル(9.11),加法モデル(9.10),最小モデル
定数,3.091,1.754,3.507
SSM,-0.318,1.694,-
NOD,-0.147,1.302,-
IND,-0.693,0.499,-
TNK,-2.398,0.444,-
EXT,-0.788,1.201,-
SSM*TNK,3.614,-,-
SSM*EXT,2.761,-,-
NOD*TNK,2.950,-,-
NOD*EXT,2.134,-,-


In [31]:
dict_coef = pd.DataFrame(
    b_additive, index=["定数", "SSM", "NOD", "IND", "TNK", "EXT"], columns=["係数"]
).to_dict()["係数"]
dict_coef

{'定数': 1.7544036826842877,
 'SSM': 1.6939953004621635,
 'NOD': 1.3019532126861397,
 'IND': 0.49899116611898775,
 'TNK': 0.4439313889359613,
 'EXT': 1.2010272940961793}

In [104]:
# 各セルにおける期待度数
mu = np.array(
    [
        [
            dict_coef["定数"],
            dict_coef["定数"] + dict_coef["TNK"],
            dict_coef["定数"] + dict_coef["EXT"],
        ],
        [
            dict_coef["定数"] + dict_coef["SSM"],
            dict_coef["定数"] + dict_coef["SSM"] + dict_coef["TNK"],
            dict_coef["定数"] + dict_coef["SSM"] + dict_coef["EXT"],
        ],
        [
            dict_coef["定数"] + dict_coef["NOD"],
            dict_coef["定数"] + dict_coef["NOD"] + dict_coef["TNK"],
            dict_coef["定数"] + dict_coef["NOD"] + dict_coef["EXT"],
        ],
        [
            dict_coef["定数"] + dict_coef["IND"],
            dict_coef["定数"] + dict_coef["IND"] + dict_coef["TNK"],
            dict_coef["定数"] + dict_coef["IND"] + dict_coef["EXT"],
        ],
    ]
)
mu = np.exp(mu)

# 各セルの割合
theta_jk = mu / mu.sum().astype(int)  # これは行と列が独立ではない時のthetaの値

# 各セルの階乗計算
list_tmp1 = []
for j in range(df.shape[0]):
    list_tmp2 = []
    for k in range(df.shape[1]):
        list_tmp2.append(sum([np.log(i) for i in range(1, df.values[j][k])]))
    list_tmp1.append(list_tmp2)
list_tmp1 = np.array(list_tmp1)

# 対数尤度
log_likelihood = (
    sum([np.log(i) for i in range(1, 401)])
    + (df.values * np.log(theta_jk) - list_tmp1).sum()
)
log_likelihood

-14.650327997182103